### RNN (Recurrent Neural Network)

An RNN (Recurrent Neural Network) is a type of artificial neural network designed for processing sequential data. Unlike traditional feedforward neural networks, RNNs have connections that form directed cycles, allowing them to maintain a 'memory' of previous inputs in the sequence. This makes them particularly well-suited for tasks where context and order are important.<br/>

RNNs work by taking an input vector at each time step and combining it with the hidden state from the previous time step to produce a new hidden state. This hidden state is then used to make predictions or generate outputs.

![image.png](./65363898_image.png)

The diagram illustrates the computational structure of a **Recurrent Neural Network (RNN)** by showing both its **folded** (compact) and **unfolded** (time-expanded) forms. 

1. Folded (compact) representation – left side

The left block shows an RNN cell **h** with three kinds of connections:

* **Input connection (U):**
  The input at time *t*, denoted **X**, enters the RNN cell via a weight matrix **U**.

* **Recurrent connection (V):**
  The hidden state from the previous time step is fed back into the cell. This feedback loop lets the network maintain memory over time.
  The hidden-to-hidden weight matrix is **V**.

* **Output connection (W):**
  The hidden state produces an output **L** through a weight matrix **W**.

This diagram expresses the idea that the RNN maintains a **single state h**, updated repeatedly over time.

2. Unfolded (expanded) representation – right side

On the right, the RNN is **unrolled** across time steps. This visualizes how the same cell is reused at every time index: ${t-1}$, ${t}$, ${t+1}$.

* Each hidden state ( h_t ) depends on both the current input ( X_t ) and the previous hidden state ( h_{t-1} ).
* The same parameters **U**, **V**, and **W** are reused at every time step. This “weight sharing” is a defining property of RNNs.

3. Interpretation

* **Temporal dependency:**
  The recurrent connection allows information to flow forward through time so the model can retain memory.

* **Backpropagation through time (BPTT):**
  During training, the unfolded network is used to compute gradients across all time steps.

### Limitations of RNNs

However, standard RNNs can suffer from 2 main issues:

-  **vanishing and exploding gradients**: When an RNN learns, it must decide how much past information to keep and how much to forget.<br/>
  To do this, the training algorithm sends a signal (the gradient) backward in time telling the network:<br/>
  “This old memory was important—strengthen it.”<br/>
  or<br/>
  “This old memory was not useful—reduce it.”<br/>
  The problems arise because this teaching signal gets weaker or stronger as it moves backward through many steps.<br/>When training RNNs with long sequences, gradients that propagate back through many steps tend to vanish or explode, making it hard to learn long range dependencies.<br/>
  
  - If the gradients grow very large (`explode`), the model's weights can be updated with excessively large values => Some past information gets amplified too much, the RNN becomes “over-sensitive” to some past events, causing erratic behavior  <br/>
  - Conversely gradients can become very small (`vanish`), making it difficult for the model to learn from earlier time steps => The network cannot learn long-term relationships (it forgets them almost immediately), only recently seen information is retained and affects decisions.<br/>
  - **sequential processing**: RNNs process tokens one by one. This sequential nature makes training slow and prevents efficient parallelization on modern hardware. As models grew larger and datasets exploded, this became a severe bottleneck.

In [ ]:
import numpy as np

# Simple RNN for sentiment analysis
# Shows sequential processing and memory (hidden state)

class SimpleRNN:
    def __init__(self, input_size, hidden_size, output_size):
        # Pre-set weights to show meaningful patterns
        # In practice, these would be learned through training
        
        # Wxh: Input to Hidden (shape: hidden_size × input_size = 2 × 3)
        # We have 2 ROWS because hidden_size=2 (2 hidden neurons)
        # We have 3 COLUMNS because input_size=3 (3 words in vocab)
        # Each row represents one hidden neuron's weights for all input words
        self.Wxh = np.array([
            # Row 1: Hidden neuron #1's response to [good, bad, movie]
            [2.0, -2.0, 0.1],   # Fires strongly for "good", negatively for "bad", neutral for "movie"
            # Row 2: Hidden neuron #2's response to [good, bad, movie]
            [1.5, -1.5, 0.2],   # Similar pattern but different sensitivity
        ])
        
        # Whh: Hidden to Hidden, maintain memory across steps (shape: hidden_size × hidden_size = 2 × 2)
        # This is the MEMORY component - how previous hidden state affects current
        # allows the RNN to remember previous words
        # changes as each word is processed
        # The final sentiment depends on ALL words seen so far        
        # We have 2 ROWS and 2 COLUMNS because hidden_size=2
        self.Whh = np.array([
            [0.5, 0.3],   # How previous h[0] and h[1] affect current h[0]
            [0.3, 0.5]    # How previous h[0] and h[1] affect current h[1]
        ])
        
        # Why: Hidden to Output (shape: output_size × hidden_size = 2 × 2)
        # We have 2 ROWS because output_size=2 (positive/negative sentiment)
        # We have 2 COLUMNS because hidden_size=2
        self.Why = np.array([
            [1.0, 0.8],    # Row 1: Weights to compute POSITIVE score
            [-1.0, -0.8]   # Row 2: Weights to compute NEGATIVE score
        ])
        
        # Biases: Allow shifting the activation (like y-intercept in y=mx+b)
        # Set to zero here for simplicity, but could be trained
        self.bh = np.zeros((hidden_size, 1))  # Bias for hidden layer
        self.by = np.zeros((output_size, 1))  # Bias for output layer
        
    def forward(self, inputs, h_prev):
        """
        Process one word at a time (sequential!)
        h_prev: previous hidden state (the "memory")
        """
        # Combine current input with previous memory
        # h is the RNN's hidden state (memory): 
        # 1) updated each step 
        # 2) carries info from previous words 
        # 3) allows later words to be interpreted in context of earlier words
        h = np.tanh(np.dot(self.Wxh, inputs) + np.dot(self.Whh, h_prev) + self.bh)
        y = np.dot(self.Why, h) + self.by
        return y, h  # Return output and new hidden state (updated memory)

# Demo: Classify sentiment of "good movie" vs "bad movie"
vocab = {"good": 0, "bad": 1, "movie": 2}
vocab_size = len(vocab)

# One-hot encode words
def encode_word(word):
    vec = np.zeros((vocab_size, 1))
    vec[vocab[word]] = 1
    return vec

# Initialize RNN
rnn = SimpleRNN(input_size=vocab_size, hidden_size=2, output_size=2)

# Process sentence word by word
sentence = ["good", "movie"]
print("Processing sentence:", " ".join(sentence))
print("\nSequential Processing (one word at a time):\n")

h = np.zeros((2, 1))  # Initial hidden state (empty memory)
for i, word in enumerate(sentence):
    print(f"Step {i+1}: Processing word '{word}'")
    x = encode_word(word)
    y, h = rnn.forward(x, h)
    
    print(f"  Hidden state (memory): [{h[0,0]:.3f}, {h[1,0]:.3f}]")
    print(f"  Output scores: positive={y[0,0]:.3f}, negative={y[1,0]:.3f}")
    sentiment = "POSITIVE" if y[0] > y[1] else "NEGATIVE"
    print(f"  → Current prediction: {sentiment}")
    print()

# The final output uses context from ALL previous words!
sentiment = "POSITIVE" if y[0] > y[1] else "NEGATIVE"
print(f"Final prediction: {sentiment}")
print("\n" + "="*60)
print("Key insight: The hidden state carries information from")
print("previous words, building context as we process the sequence!")
print("="*60)

# Compare with different first word
print("\n\nNow let's try 'bad movie' to see context in action:\n")
sentence2 = ["bad", "movie"], 
print("Processing sentence:", " ".join(sentence2))
print()

h = np.zeros((2, 1))
for i, word in enumerate(sentence2):
    print(f"Step {i+1}: Processing word '{word}'")
    x = encode_word(word)
    y, h = rnn.forward(x, h)
    
    print(f"  Hidden state (memory): [{h[0,0]:.3f}, {h[1,0]:.3f}]")
    print(f"  Output scores: positive={y[0,0]:.3f}, negative={y[1,0]:.3f}")
    sentiment = "POSITIVE" if y[0] > y[1] else "NEGATIVE"
    print(f"  → Current prediction: {sentiment}")
    print()

sentiment = "POSITIVE" if y[0] > y[1] else "NEGATIVE"
print(f"Final prediction: {sentiment}")
print("\nNotice: Same word 'movie' produces DIFFERENT outputs")
print("depending on the context from earlier words!")

# Educational explanation of dimensions
print("\n" + "="*60)
print("UNDERSTANDING THE MATRIX DIMENSIONS")
print("="*60)
print("\n1. Why 2 rows in each matrix?")
print("   Because hidden_size=2, we have 2 hidden neurons.")
print("   Think of them as 2 'memory cells' that each learn")
print("   different patterns from the sequence.")
print("\n2. The forward pass computation:")
print("   h = tanh(Wxh @ input + Whh @ h_prev + bh)")
print("   ")
print("   Example when processing 'good' [1,0,0]:")
print("   ")
print("   Wxh @ [1,0,0] = [2.0, -2.0, 0.1]  @  [1]   =  [2.0]")
print("                   [1.5, -1.5, 0.2]     [0]      [1.5]")
print("                                        [0]")
print("   ")
print("   This extracts the FIRST COLUMN of Wxh (weights for 'good')")
print("   ")
print("3. Why do we need multiple neurons?")
print("   - Neuron 1 might learn: 'detect positive words'")
print("   - Neuron 2 might learn: 'detect negation patterns'")
print("   - More neurons = more complex patterns")
print("\n4. About biases (bh and by):")
print("   Set to zero here for simplicity. In training, they would")
print("   be adjusted to help the network make better predictions.")
print("   Think of them as the y-intercept: they shift the activation.")
print("="*60)

Processing sentence: good movie

Sequential Processing (one word at a time):

Step 1: Processing word 'good'
  Hidden state (memory): [0.964, 0.905]
  Output scores: positive=1.688, negative=-1.688
  → Current prediction: POSITIVE

Step 2: Processing word 'movie'
  Hidden state (memory): [0.693, 0.736]
  Output scores: positive=1.282, negative=-1.282
  → Current prediction: POSITIVE

Final prediction: POSITIVE

Key insight: The hidden state carries information from
previous words, building context as we process the sequence!


Now let's try 'bad movie' to see context in action:

Processing sentence: movie bad

Step 1: Processing word 'movie'
  Hidden state (memory): [0.100, 0.197]
  Output scores: positive=0.258, negative=-0.258
  → Current prediction: POSITIVE

Step 2: Processing word 'bad'
  Hidden state (memory): [-0.955, -0.879]
  Output scores: positive=-1.659, negative=1.659
  → Current prediction: NEGATIVE

Final prediction: NEGATIVE

Notice: Same word 'movie' produces DIFFERENT

In [12]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# Simple character-level RNN using PyTorch
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        """
        Simple RNN implementation using PyTorch
        input_size: size of vocabulary
        hidden_size: number of hidden units
        output_size: size of vocabulary (same as input for char prediction)
        """
        super(SimpleRNN, self).__init__()
        
        self.hidden_size = hidden_size
        
        # RNN layer (handles W_xh and W_hh internally)
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        
        # Output layer (W_hy)
        self.fc = nn.Linear(hidden_size, output_size)
        
    def forward(self, x, h_prev=None):
        """
        Forward pass through the RNN
        x: input tensor of shape (batch_size, seq_len, input_size)
        h_prev: previous hidden state
        Returns: output and hidden states
        """
        if h_prev is None:
            h_prev = torch.zeros(1, x.size(0), self.hidden_size)
        
        # RNN forward pass (memory happens here!)
        out, h = self.rnn(x, h_prev)
        
        # Pass through output layer
        out = self.fc(out)
        
        return out, h

# Example: Learn a simple sequence
print("=== PyTorch RNN: Memory Demo ===\n")

# Create a simple text dataset
text = "hello world"
chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_ix = {ch: i for i, ch in enumerate(chars)}
ix_to_char = {i: ch for i, ch in enumerate(chars)}

print(f"Text: '{text}'")
print(f"Vocabulary: {chars}")
print(f"Vocab size: {vocab_size}\n")

# Create RNN
hidden_size = 10
rnn = SimpleRNN(input_size=vocab_size, hidden_size=hidden_size, output_size=vocab_size)

print(f"Model architecture:")
print(rnn)
print()

# Convert text to indices
data = [char_to_ix[ch] for ch in text]
print(f"Character sequence as indices: {data[:20]}...\n")

# Demonstrate forward pass showing memory flow
print("=== Forward Pass (Memory Flow) ===")

# Process first few characters to show memory update
h = None  # Start with no hidden state (will be initialized to zeros)

for i in range(3):
    char = text[i]
    char_idx = char_to_ix[char]
    
    # Create one-hot encoded input
    x = torch.zeros(1, 1, vocab_size)  # (batch=1, seq_len=1, vocab_size)
    x[0, 0, char_idx] = 1
    
    # Forward pass
    out, h = rnn(x, h)
    
    # Get probabilities
    probs = torch.softmax(out[0, 0], dim=0)
    predicted_idx = torch.argmax(probs).item()
    predicted_char = ix_to_char[predicted_idx]
    
    print(f"Step {i+1}: Input char '{char}' (index {char_idx})")
    print(f"  Hidden state shape: {h.shape}")
    print(f"  Hidden state values (first 5): {h[0, 0, :5].detach().numpy()}")
    print(f"  Predicted: '{predicted_char}' (confidence: {probs[predicted_idx]:.2%})")
    print(f"  Memory carries forward to next step!\n")

print("=== Key Concept ===")
print("PyTorch RNN internally computes:")
print("h[t] = tanh(W_ih * x[t] + b_ih + W_hh * h[t-1] + b_hh)")
print("                              ^^^^^^^^^^^^^^^^^^^^")
print("                              This is the MEMORY mechanism!\n")

# Visualize memory flow through the sequence
print("\n=== Memory Flow Through Sequence ===")
print("Processing: 'hello world'")
print("-" * 100)

# Process the sequence and track hidden states
h = None
memory_flow = []

for i, char in enumerate(text[:11]):  # First "hello world"
    char_idx = char_to_ix[char]
    
    # One-hot encode
    x = torch.zeros(1, 1, vocab_size)
    x[0, 0, char_idx] = 1
    
    # Forward pass
    out, h = rnn(x, h)
    
    # Get prediction
    probs = torch.softmax(out[0, 0], dim=0)
    predicted_idx = torch.argmax(probs).item()
    predicted_char = ix_to_char[predicted_idx]
    
    # Calculate hidden state magnitude
    h_norm = torch.norm(h).item()
    
    memory_flow.append({
        'step': i,
        'input': char,
        'h_norm': h_norm,
        'prediction': predicted_char,
        'confidence': probs[predicted_idx].item(),
        'h_values': h[0, 0, :3].detach().numpy()
    })

# Print table
print(f"{'Step':<6} {'Input':<8} {'Memory Mag.':<15} {'Prediction':<12} {'Confidence':<12} {'Hidden (first 3)':<30}")
print("=" * 100)

for row in memory_flow:
    h_vals = ', '.join([f"{v:6.3f}" for v in row['h_values']])
    print(f"{row['step']:<6} {repr(row['input']):<8} {row['h_norm']:<15.6f} {repr(row['prediction']):<12} {row['confidence']:<12.2%} [{h_vals}]")

print("-" * 100)

# Memory persistence visualization
print("\n\n=== Memory Persistence Visualization ===")
print(f"{'Step':<8} {'Char':<8} {'Memory Magnitude':<20} {'Visual':<40}")
print("-" * 80)

max_norm = max(row['h_norm'] for row in memory_flow)
for row in memory_flow:
    bar_length = int((row['h_norm'] / max_norm) * 30)
    bar = '█' * bar_length
    print(f"{row['step']:<8} {repr(row['input']):<8} {row['h_norm']:<20.6f} {bar}")

print("-" * 80)

# Show sequence processing with batch
print("\n\n=== Batch Sequence Processing ===")
print("Processing entire 'hello world' as one sequence\n")

# Create sequence tensor
sequence = "hello world"
seq_indices = [char_to_ix[ch] for ch in sequence]

# One-hot encode the entire sequence
x_seq = torch.zeros(1, len(sequence), vocab_size)
for t, idx in enumerate(seq_indices):
    x_seq[0, t, idx] = 1

# Forward pass through entire sequence
h = None
out_seq, h_final = rnn(x_seq, h)

print(f"Input sequence shape: {x_seq.shape} (batch=1, seq_len={len(sequence)}, vocab_size={vocab_size})")
print(f"Output sequence shape: {out_seq.shape}")
print(f"Final hidden state shape: {h_final.shape}\n")

# Show predictions for each time step
print(f"{'Time':<6} {'Input':<10} {'→':<3} {'Predicted':<12} {'Confidence':<12}")
print("=" * 50)

for t in range(len(sequence)):
    probs = torch.softmax(out_seq[0, t], dim=0)
    predicted_idx = torch.argmax(probs).item()
    predicted_char = ix_to_char[predicted_idx]
    confidence = probs[predicted_idx].item()
    
    print(f"t={t:<4} {repr(sequence[t]):<10} → {repr(predicted_char):<12} {confidence:<12.2%}")

print("=" * 50)

print("\n=== PyTorch RNN Benefits ===")
print("✓ Automatic gradient computation (backpropagation through time)")
print("✓ Efficient batched operations")
print("✓ GPU acceleration support")
print("✓ Built-in weight initialization")
print("✓ Easy to extend to LSTM/GRU variants")

# Show how to access RNN weights
print("\n\n=== RNN Internal Weights ===")
print("Weight matrices that enable memory:")
for name, param in rnn.rnn.named_parameters():
    print(f"  {name}: shape {param.shape}")

print("\nNote: weight_ih = W_xh (input to hidden)")
print("      weight_hh = W_hh (hidden to hidden) ← This is the MEMORY weight!")

=== PyTorch RNN: Memory Demo ===

Text: 'hello world'
Vocabulary: [' ', 'd', 'e', 'h', 'l', 'o', 'r', 'w']
Vocab size: 8

Model architecture:
SimpleRNN(
  (rnn): RNN(8, 10, batch_first=True)
  (fc): Linear(in_features=10, out_features=8, bias=True)
)

Character sequence as indices: [3, 2, 4, 4, 5, 0, 7, 5, 6, 4, 1]...

=== Forward Pass (Memory Flow) ===
Step 1: Input char 'h' (index 3)
  Hidden state shape: torch.Size([1, 1, 10])
  Hidden state values (first 5): [-0.13062131 -0.37338656 -0.2353717   0.06485475  0.2345997 ]
  Predicted: 'd' (confidence: 15.83%)
  Memory carries forward to next step!

Step 2: Input char 'e' (index 2)
  Hidden state shape: torch.Size([1, 1, 10])
  Hidden state values (first 5): [-0.5877452  -0.46572325 -0.75946605 -0.16404194 -0.34474465]
  Predicted: 'h' (confidence: 17.35%)
  Memory carries forward to next step!

Step 3: Input char 'l' (index 4)
  Hidden state shape: torch.Size([1, 1, 10])
  Hidden state values (first 5): [-0.2445206  -0.06282393 -0.345